In [1]:
# Clear any existing GPU memory
import torch
import gc

# Check if CUDA is available before accessing GPU
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    
    # Check available memory
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"Available: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1e9:.2f} GB")
    print(f"CUDA Device: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  CUDA is not available. Make sure you're using a GPU-enabled Kaggle notebook.")
    print("   Go to: Settings → Accelerator → Select GPU (T4 or P100)")
    print("   Then restart the kernel and run this cell again.")
    gc.collect()  # Still do garbage collection even without GPU

GPU Memory: 15.83 GB
Available: 15.83 GB


In [ ]:
# RUN THIS ONCE, THEN RESTART THE SESSION AND THEN DO NOT RUN THIS CELL AGAIN
# Install required packages
!pip install -q -U accelerate transformers flask flask-cors pyngrok

In [ ]:
import torch
import re
import unicodedata
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
import threading

In [4]:
# PASTE YOUR NGROK TOKEN BELOW inside the quotes
NGROK_AUTH_TOKEN = "36ZSxCZUYnZVSpKzp3tMXXVhhKW_5dcPZBWDWP83woV5LSDVE"
MODEL_ID = "large-traversaal/Alif-1.0-8B-Instruct"

In [ ]:
# CRITICAL: Clear GPU memory before loading model
print("🧹 Clearing GPU memory...")
print("\n⚠️  IMPORTANT: If you get OutOfMemoryError, RESTART THE KERNEL first!")
print("   Go to: Runtime -> Restart runtime (or Kernel -> Restart)")
print("   Then run cells from the beginning\n")

# More aggressive memory clearing
try:
    # Clear any existing models/variables
    if 'model' in globals():
        print("   Removing existing model...")
        del model
    if 'chatbot' in globals():
        print("   Removing existing pipeline...")
        del chatbot
    if 'tokenizer' in globals():
        print("   Removing existing tokenizer...")
        del tokenizer
    
    # Force garbage collection multiple times
    for _ in range(3):
        gc.collect()
    
    # Clear CUDA cache aggressively
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()  # Wait for all operations to complete
        # Reset peak memory stats
        torch.cuda.reset_peak_memory_stats()
    
    print("✅ GPU memory cleared")
except Exception as e:
    print(f"⚠️  Warning during cleanup: {e}")
    # Still try to clear cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

# Check GPU memory
if torch.cuda.is_available():
    print(f"\n💾 GPU Memory Status:")
    print(f"   Total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    print(f"   Allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print(f"   Reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
    print(f"   Free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved(0)) / 1024**3:.2f} GB")

print("\n⏳ Loading Model (optimized for quality)...")
print("   This may take a few minutes...")

# Load tokenizer first (lightweight)
print("\n📝 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print("✅ Tokenizer loaded")

# Load model with float16 for memory efficiency while maintaining quality
print("\n🤖 Loading model with float16 (memory-efficient, quality-focused)...")

try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map="auto",
        torch_dtype=torch.float16,  # Use float16 for memory efficiency
        low_cpu_mem_usage=True,
        trust_remote_code=True
    )
    print("✅ Model loaded successfully!")
except RuntimeError as e:
    # PyTorch raises RuntimeError for CUDA OOM, check if it's an OOM error
    if "out of memory" in str(e).lower() or "CUDA" in str(e) or "OOM" in str(e):
        print(f"\n❌ Out of Memory Error: {e}")
        print("\n🔧 TROUBLESHOOTING STEPS:")
        print("1. RESTART THE KERNEL (Runtime -> Restart runtime)")
        print("2. Run all cells from the beginning")
        print("3. If still failing, try the fallback method below")
        print("\n🔄 Trying fallback: Loading with memory limits...")
        
        # Clear everything again more aggressively
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
        for _ in range(3):
            gc.collect()
        
        try:
            # Try with explicit memory limits
            model = AutoModelForCausalLM.from_pretrained(
                MODEL_ID,
                device_map="auto",
                torch_dtype=torch.float16,
                low_cpu_mem_usage=True,
                max_memory={0: "13GiB"},  # Reserve some memory
                trust_remote_code=True
            )
            print("✅ Model loaded with memory limits (fallback mode)")
        except RuntimeError as oom2:
            # Check if it's an OOM error
            if "out of memory" in str(oom2).lower() or "CUDA" in str(oom2) or "OOM" in str(oom2):
                print(f"\n❌ Still out of memory: {oom2}")
                print("\n🚨 CRITICAL: You MUST restart the kernel!")
                print("   Go to: Runtime -> Restart runtime")
                print("   Then run all cells from the beginning")
                print("\n💡 The GPU memory is completely full. Restarting will clear it.")
            raise
    else:
        # Not an OOM error, re-raise the original exception
        raise
except Exception as e:
    # Check if it's an OOM error
    if "out of memory" in str(e).lower() or "CUDA" in str(e) or "OOM" in str(e):
        print(f"\n❌ Out of Memory Error: {e}")
        print("\n🚨 CRITICAL: You MUST restart the kernel!")
        print("   Go to: Runtime -> Restart runtime")
        print("   Then run all cells from the beginning")
        raise
    else:
        print(f"\n❌ Error loading model: {e}")
    print("\n🔄 Trying fallback: Loading with float16 only (no quantization)...")
    # Fallback: Try without quantization but with memory optimizations
    torch.cuda.empty_cache()
    gc.collect()
    
    try:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            device_map="auto",
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
            max_memory={0: "14GiB"},  # Reserve some memory
            trust_remote_code=True
        )
        print("✅ Model loaded with float16 (fallback mode)")
    except Exception as e2:
        print(f"\n❌ Fallback also failed: {e2}")
        print("\n🚨 Please restart the kernel and try again!")
        raise

# Check memory after loading
if torch.cuda.is_available():
    print(f"\n💾 GPU Memory After Loading:")
    print(f"   Allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print(f"   Reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
    free_mem = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved(0)) / 1024**3
    print(f"   Free: {free_mem:.2f} GB")
    if free_mem < 1.0:
        print("⚠️  Warning: Very little free memory remaining!")


⏳ Loading Model... this may take a few minutes...


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
# Initialize Pipeline with memory-efficient settings
print("\n🔧 Initializing text generation pipeline...")
try:
    chatbot = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        device_map="auto",
        return_full_text=False,
        torch_dtype=torch.float16  # Use float16 for memory efficiency
    )
    print("✅ Pipeline initialized successfully!")
    
    # Final memory check
    if torch.cuda.is_available():
        print(f"\n💾 Final GPU Memory Status:")
        print(f"   Allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
        print(f"   Reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
        free_mem = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved(0)) / 1024**3
        print(f"   Free: {free_mem:.2f} GB")
        
    print("\n✅ Model and Pipeline Ready for Use!")
except Exception as e:
    print(f"❌ Error initializing pipeline: {e}")
    raise


In [ ]:
# ==========================================
# FLASK API SETUP
# ==========================================
app = Flask(__name__)
CORS(app)  # Enable Cross-Origin Resource Sharing for your local frontend


In [ ]:
# Default prompt (fallback only - prompts are now provided from frontend)
# This is used only if no custom_prompt is sent from the frontend
STATIC_PROMPT_CONTEXT = """You are an expert Urdu text correction system. Your task is to produce the BEST and MOST ACCURATE transcription by combining the 4 ASR transcriptions above into one perfect Urdu sentence.

Quality Guidelines:
1. Use words where 2+ models agree (confidence >0.60) - this indicates high reliability
2. Prefer highest confidence words when models disagree - choose the most confident option
3. Fix ALL spelling errors - ensure proper Urdu orthography
4. Ensure proper Urdu grammar and sentence structure
5. Use natural, simple, and clear Urdu language
6. Maintain the original meaning and intent
7. Ensure complete words (no partial words or truncations)

Output: Only the BEST corrected Urdu sentence with proper punctuation (۔). Use clean, standard Urdu characters only. Focus on accuracy and quality above all else."""

# Default prompt (fallback only - prompts are now provided from frontend)
# This is used only if no custom_prompt is sent from the frontend
STATIC_PROMPT_CONTEXT = """You are an expert Urdu text correction system. Your task is to produce the BEST and MOST ACCURATE transcription by combining 4 ASR transcriptions into one perfect Urdu sentence.

Quality Guidelines:
1. Use words where 2+ models agree (confidence >0.60) - this indicates high reliability
2. Prefer highest confidence words when models disagree - choose the most confident option
3. Fix ALL spelling errors - ensure proper Urdu orthography
4. Ensure proper Urdu grammar and sentence structure
5. Use natural, simple, and clear Urdu language
6. Maintain the original meaning and intent
7. Ensure complete words (no partial words or truncations)

Output: Only the BEST corrected Urdu sentence with proper punctuation (۔). Use clean, standard Urdu characters only. Focus on accuracy and quality above all else.

Input:

"""

In [ ]:
# Remove existing route if it exists (to allow re-running the cell)
# This prevents AssertionError when re-running the cell in Jupyter
try:
    if 'generate_correction' in app.view_functions:
        del app.view_functions['generate_correction']
        print("✅ Removed existing generate_correction route")
    # Also remove from url_map if possible
    try:
        rules_to_remove = [rule for rule in app.url_map.iter_rules() 
                          if rule.endpoint == 'generate_correction']
        for rule in rules_to_remove:
            app.url_map._rules.remove(rule)
            if rule.endpoint in app.url_map._rules_by_endpoint:
                del app.url_map._rules_by_endpoint[rule.endpoint]
    except Exception as e:
        print(f"⚠️ Could not clean url_map: {e}")
        pass  # If url_map cleanup fails, continue anyway
except Exception as e:
    print(f"⚠️ Route cleanup warning: {e}")
    pass

# Register the /correct endpoint
@app.route('/correct', methods=['POST'])
def generate_correction():
    """Endpoint for generating corrected Urdu text from ASR hypotheses"""
    try:
        # Check if model is loaded
        try:
            if chatbot is None:
                return jsonify({"error": "Model not loaded. Please load the model first."}), 500
        except NameError:
            return jsonify({"error": "Model not loaded. Please load the model first."}), 500
        
        data = request.json
        if not data:
            return jsonify({"error": "No JSON data received"}), 400
            
        hypotheses = data.get('hypotheses', [])
        custom_prompt = data.get('custom_prompt', '')

        if not hypotheses or len(hypotheses) != 4:
            return jsonify({"error": "Please provide exactly 4 hypotheses from ASR models."}), 400

        # Clear CUDA cache before inference to help prevent OOM
        try:
            import torch
            torch.cuda.empty_cache()
        except:
            pass

        # IMPORTANT: Transcriptions go FIRST, then the prompt/instructions
        # This ensures the model sees the data before receiving instructions
        
        # Construct the hypotheses section (transcriptions first)
        dynamic_input = ""
        for i, hyp in enumerate(hypotheses):
            dynamic_input += f"H{i+1}: {hyp}\n"
        
        # Add a separator after transcriptions
        dynamic_input += "\n"

        # Use custom prompt if provided, otherwise use default
        if custom_prompt and custom_prompt.strip():
            # Check for various placeholder formats
            if "{hypotheses}" in custom_prompt:
                # Replace {hypotheses} placeholder with formatted transcriptions
                full_prompt = custom_prompt.replace("{hypotheses}", dynamic_input)
            elif "{predictions[0]}" in custom_prompt or "{predictions[1]}" in custom_prompt or "{predictions[2]}" in custom_prompt or "{predictions[3]}" in custom_prompt:
                # Replace numbered prediction placeholders (for Urdu prompt format)
                # This format allows transcriptions to be embedded in the prompt structure
                full_prompt = custom_prompt
                for i, hyp in enumerate(hypotheses):
                    placeholder = f"{{predictions[{i}]}}"
                    if placeholder in full_prompt:
                        full_prompt = full_prompt.replace(placeholder, hyp)
            else:
                # Transcriptions FIRST, then prompt/instructions
                full_prompt = f"{dynamic_input}{custom_prompt.strip()}"
        else:
            # Use default prompt: transcriptions FIRST, then instructions
            full_prompt = f"{dynamic_input}{STATIC_PROMPT_CONTEXT}"
        
        # Ensure prompt doesn't end with just "Output:" - add a space or newline to encourage generation
        # Some models stop generating if the prompt ends with "Output:" without continuation
        if full_prompt.rstrip().endswith("Output:") or full_prompt.rstrip().endswith("Output"):
            full_prompt = full_prompt.rstrip() + "\n"
        
        # Also ensure the prompt ends properly to encourage generation
        if not full_prompt.endswith("\n") and not full_prompt.endswith(" "):
            full_prompt = full_prompt + " "

        # Debug: Print the prompt being sent to the model
        print("=" * 80)
        print("📤 PROMPT SENT TO MODEL:")
        print("=" * 80)
        print(full_prompt)
        print("=" * 80)
        print(f"Prompt length: {len(full_prompt)} characters")
        print("=" * 80)
        
        # Run Inference with optimized parameters for BEST QUALITY transcription
        # Quality optimizations:
        # - temperature=0.2: Balanced (not too deterministic, allows quality variations)
        # - top_p=0.85: Nucleus sampling (focuses on high-probability quality tokens)
        # - top_k=50: Limits to top 50 tokens (quality-focused selection)
        # - repetition_penalty=1.1: Reduces repetition, improves quality
        # - max_new_tokens=250: Ensures complete sentences
        # - min_length=15: Forces minimum quality output length
        # Note: Some parameters might not be supported by all pipeline versions
        try:
            response = chatbot(
                full_prompt,
                max_new_tokens=250,  # Increased for complete sentences
                min_length=15,  # Force minimum output length for quality
                do_sample=True,
                temperature=0.2,  # Balanced: deterministic but allows quality variations
                top_p=0.85,  # Nucleus sampling for quality (focuses on high-probability tokens)
                top_k=50,  # Limit to top 50 tokens for quality
                repetition_penalty=1.1,  # Higher to avoid repetition and improve quality
                no_repeat_ngram_size=3,
                pad_token_id=tokenizer.eos_token_id  # Ensure proper padding
            )
        except TypeError as te:
            # If some parameters are not supported, try with quality-focused basic parameters
            print(f"Warning: Some parameters not supported, using quality-focused basic parameters. Error: {te}")
            try:
                response = chatbot(
                    full_prompt,
                    max_new_tokens=250,
                    min_length=15,
                    do_sample=True,
                    temperature=0.2,  # Quality-focused temperature
                    top_p=0.85,  # Nucleus sampling for quality
                    repetition_penalty=1.1
                )
            except TypeError as te2:
                # Fallback to minimal but still quality-focused parameters
                print(f"Warning: Advanced parameters not supported, using minimal quality-focused. Error: {te2}")
                response = chatbot(
                    full_prompt,
                    max_new_tokens=250,
                    do_sample=True,
                    temperature=0.2  # Still quality-focused even in minimal fallback
                )
        
        # Debug: Print the raw response structure
        print("=" * 80)
        print("📥 RAW RESPONSE STRUCTURE:")
        print("=" * 80)
        print(f"Response type: {type(response)}")
        print(f"Response: {response}")
        if isinstance(response, list) and len(response) > 0:
            print(f"Response[0] type: {type(response[0])}")
            print(f"Response[0] keys: {response[0].keys() if isinstance(response[0], dict) else 'N/A'}")
        print("=" * 80)

        if not response:
            return jsonify({"error": "Model returned None response"}), 500
            
        if not isinstance(response, list) or len(response) == 0:
            return jsonify({"error": f"Model returned invalid response format: {type(response)}"}), 500

        if "generated_text" not in response[0]:
            return jsonify({"error": f"Response missing 'generated_text' key. Response: {response[0]}"}), 500

        raw_response = response[0].get("generated_text", "").strip()
        
        # If we got an empty or very short response, try with different parameters
        # Check if response is just punctuation or too short
        is_minimal = (not raw_response or len(raw_response.strip()) == 0 or 
                      (len(raw_response.strip()) <= 3 and all(c in '۔،؟!.,;:!?' or c.isspace() for c in raw_response.strip())))
        
        if is_minimal:
            print("⚠️ First attempt returned minimal/empty response. Trying with quality-focused retry parameters...")
            try:
                # Quality-focused retry: higher tokens, balanced temperature for best quality
                retry_response = chatbot(
                    full_prompt,
                    max_new_tokens=350,  # Increased for complete, high-quality sentences
                    min_length=25,  # Force longer minimum for quality
                    do_sample=True,
                    temperature=0.25,  # Slightly higher but still quality-focused
                    top_p=0.9,  # Slightly wider for more options
                    top_k=60,  # More token options for quality
                    repetition_penalty=1.15,  # Higher to avoid repetition and improve quality
                    pad_token_id=tokenizer.eos_token_id
                )
                retry_text = retry_response[0].get("generated_text", "").strip()
                if retry_text and len(retry_text) > 0:
                    # Check if retry got more meaningful content with quality validation
                    urdu_in_retry = sum(1 for c in retry_text if 0x0600 <= ord(c) <= 0x06FF)
                    # Quality check: ensure we have substantial Urdu content
                    if urdu_in_retry > 3 or len(retry_text) > 8:
                        print(f"✅ Quality retry successful! Got {len(retry_text)} characters, {urdu_in_retry} Urdu chars")
                        raw_response = retry_text
                    else:
                        print(f"⚠️ Retry still minimal quality: {len(retry_text)} chars, {urdu_in_retry} Urdu chars")
                else:
                    print(f"⚠️ Retry also returned empty: {retry_response}")
            except Exception as retry_error:
                print(f"❌ Retry failed: {retry_error}")
        
        # Check if response is still empty after retry
        if not raw_response or len(raw_response.strip()) == 0:
            print("=" * 80)
            print("❌ EMPTY RESPONSE DEBUG:")
            print(f"   Full response object: {response}")
            print(f"   Response type: {type(response)}")
            if isinstance(response, list) and len(response) > 0:
                print(f"   Response[0] keys: {response[0].keys() if isinstance(response[0], dict) else 'N/A'}")
                print(f"   Response[0] full: {response[0]}")
            print(f"   Prompt length: {len(full_prompt)}")
            print(f"   Prompt preview: {full_prompt[:200]}...")
            print("=" * 80)
            return jsonify({
                "error": "Model returned empty text",
                "raw_response": str(response),
                "debug_info": f"The model generated an empty response. Prompt length: {len(full_prompt)} chars. This may indicate the model needs different parameters or the prompt format needs adjustment.",
                "prompt_preview": full_prompt[:300]
            }), 500
        
        # Check if response is just a prefix like "Answer:" with no content
        prefix_only_patterns = [
            r'^Answer:\s*$',
            r'^Response:\s*$',
            r'^Output:\s*$',
            r'^Corrected:\s*$',
        ]
        
        is_prefix_only = False
        for pattern in prefix_only_patterns:
            if re.match(pattern, raw_response, re.IGNORECASE):
                is_prefix_only = True
                break
        
        # If we only got a prefix with no content, return helpful error
        # (Don't retry as it causes CUDA OOM errors)
        if is_prefix_only or (len(raw_response) < 5 and not any(0x0600 <= ord(c) <= 0x06FF for c in raw_response)):
            print("⚠️ Model returned only a prefix or very short response with no Urdu content.")
            print(f"   Raw response: '{raw_response}'")
            return jsonify({
                "error": "Model returned only a prefix with no content",
                "raw_response": raw_response,
                "debug_info": f"The model generated only '{raw_response}' with no actual Urdu text. This may indicate the model needs different prompt parameters or the prompt format needs adjustment."
            }), 500
        
        if not raw_response or len(raw_response.strip()) == 0:
            return jsonify({
                "error": "Model returned empty text",
                "raw_response": str(response),
                "debug_info": "The model generated an empty response"
            }), 500
        
        # Debug: Print FULL raw response to help diagnose issues
        print("=" * 80)
        print("🔍 FULL RAW MODEL RESPONSE:")
        print("=" * 80)
        print(raw_response)
        print("=" * 80)
        print(f"Full response length: {len(raw_response)} characters")
        print(f"Full prompt length: {len(full_prompt)} characters")
        print("=" * 80)
        
        # IMPORTANT: The model returns the full prompt + generated text
        # We need to remove the prompt to get only the generated part
        corrected_text = raw_response
        original_length = len(corrected_text)
        
        # Store original for fallback if removal goes wrong
        original_response = raw_response
        
        # Remove the prompt from the response if it's present
        # The prompt typically ends with "Input:\n" or the hypotheses
        prompt_removed = False
        if full_prompt in corrected_text:
            before_removal = corrected_text
            corrected_text = corrected_text.replace(full_prompt, "", 1).strip()
            # Safety check: if removal left us with almost nothing, something went wrong
            if len(corrected_text) < 3 and len(before_removal) > len(full_prompt) + 10:
                print(f"⚠️ Prompt removal left too little content. Checking if response is in raw text...")
                # Try to find actual response after prompt markers
                # Look for Urdu text after the prompt
                prompt_end_idx = before_removal.find(full_prompt) + len(full_prompt)
                if prompt_end_idx < len(before_removal):
                    after_prompt = before_removal[prompt_end_idx:].strip()
                    # Check if there's meaningful content after prompt
                    urdu_after = sum(1 for c in after_prompt if 0x0600 <= ord(c) <= 0x06FF)
                    if urdu_after > 2 or len(after_prompt) > 5:
                        corrected_text = after_prompt
                        print(f"✅ Found response after prompt: {len(corrected_text)} chars")
                    else:
                        # Maybe the response is actually just punctuation or very short
                        corrected_text = after_prompt
                        print(f"⚠️ Response after prompt is very short: '{corrected_text}'")
            else:
                print(f"✅ Removed prompt using exact match. Remaining: {len(corrected_text)} chars")
            prompt_removed = True
        else:
            # Try to find where the prompt ends by looking for the last hypothesis marker
            # This is a fallback if exact match doesn't work
            found_marker = False
            for i in range(4, 0, -1):
                marker = f"H{i}:"
                if marker in corrected_text:
                    # Find the last occurrence and take everything after it
                    last_idx = corrected_text.rfind(marker)
                    if last_idx != -1:
                        # Find the end of this line
                        line_end = corrected_text.find('\n', last_idx)
                        if line_end != -1:
                            before_removal = corrected_text
                            corrected_text = corrected_text[line_end:].strip()
                            # Safety check
                            if len(corrected_text) < 3 and len(before_removal) > 50:
                                # Try to find Urdu content after this marker
                                after_marker = before_removal[line_end:].strip()
                                urdu_after = sum(1 for c in after_marker if 0x0600 <= ord(c) <= 0x06FF)
                                if urdu_after > 2:
                                    corrected_text = after_marker
                            print(f"✅ Removed prompt using marker H{i}. Remaining: {len(corrected_text)} chars")
                            found_marker = True
                            prompt_removed = True
                            break
            if not found_marker:
                print(f"⚠️ Could not find prompt markers. Using full response: {len(corrected_text)} chars")
                # If no markers found, the response might not contain the prompt
                # Check if it looks like a direct response
                urdu_in_response = sum(1 for c in corrected_text if 0x0600 <= ord(c) <= 0x06FF)
                if urdu_in_response > 2:
                    print(f"✅ Response appears to be direct (no prompt detected). Urdu chars: {urdu_in_response}")
        
        # Remove common English prefixes that models add
        # But only if there's actual content after the prefix
        prefixes_to_remove = [
            "Answer:",
            "Response:",
            "### Response:",
            "### Answer:",
            "Output:",
            "Corrected text:",
            "Corrected:",
            "Result:",
        ]
        
        for prefix in prefixes_to_remove:
            # Only remove if there's content after the prefix
            if corrected_text.startswith(prefix):
                remaining = corrected_text[len(prefix):].strip()
                if len(remaining) > 0:  # Only remove if there's content after
                    corrected_text = remaining
                    print(f"✅ Removed prefix '{prefix}'. Remaining: {len(corrected_text)} chars")
                else:
                    print(f"⚠️ Prefix '{prefix}' found but no content after it. Keeping as-is.")
            elif prefix in corrected_text and not corrected_text.startswith(prefix):
                # If prefix appears in the middle, take everything after it
                parts = corrected_text.split(prefix, 1)
                if len(parts) > 1 and len(parts[-1].strip()) > 0:
                    corrected_text = parts[-1].strip()
                    print(f"✅ Removed prefix '{prefix}' from middle. Remaining: {len(corrected_text)} chars")
        
        print(f"📝 After prefix removal: '{corrected_text[:100]}...' (length: {len(corrected_text)})")
        
        # Take only the first line (the sentence) - but keep as fallback
        lines = corrected_text.split('\n')
        first_line = lines[0].strip() if lines else corrected_text.strip()
        print(f"📄 First line extracted: '{first_line[:100]}...' (length: {len(first_line)})")
        
        # Remove any trailing English words or explanations
        # Urdu sentences typically end with punctuation, so stop at first English word if present
        urdu_text = first_line
        # Find where Urdu text ends (if there's English after)
        for i, char in enumerate(first_line):
            if ord(char) < 128 and char.isalpha() and i > 10:  # Likely English after Urdu
                urdu_text = first_line[:i].strip()
                print(f"✂️ Removed trailing English. Urdu text: '{urdu_text[:100]}...'")
                break
        
        # Store less-cleaned version as fallback
        fallback_text = urdu_text.strip()
        print(f"💾 Fallback text stored: '{fallback_text[:100]}...' (length: {len(fallback_text)})")
        
        # Remove corrupted characters and encoding issues (light cleaning)
        corrected_text = re.sub(r'[*\\|]', '', urdu_text)
        
        # Remove any remaining confidence scores if model added them
        corrected_text = re.sub(r'\([\d.]+\)', '', corrected_text).strip()
        
        # Fix common corrupted Urdu characters
        char_fixes = {
            'ۃ': 'ہ',  # Corrupted heh to proper heh
            'ۓ': 'ے',  # Corrupted yeh to proper yeh
            'ۛ': '',   # Remove diacritic marks
            'ۘ': '',   # Remove diacritic marks
            '۟': '',   # Remove diacritic marks
            'ۙ': '',   # Remove diacritic marks
            'ۚ': '',   # Remove diacritic marks
        }
        for corrupted, correct in char_fixes.items():
            corrected_text = corrected_text.replace(corrupted, correct)
        
        # Count Urdu characters to validate content
        urdu_char_count = sum(1 for char in corrected_text if 0x0600 <= ord(char) <= 0x06FF)
        
        # Less aggressive cleaning - keep more characters
        # Only remove clearly problematic characters, keep everything else
        cleaned_chars = []
        for char in corrected_text:
            code_point = ord(char)
            # Keep Urdu/Arabic characters (0600-06FF), spaces, and punctuation
            if (0x0600 <= code_point <= 0x06FF) or char.isspace() or char in '۔،؟!.,;:!?':
                cleaned_chars.append(char)
            # Keep digits (might be part of text)
            elif char.isdigit():
                cleaned_chars.append(char)
            # Remove only clearly problematic characters (control chars, special symbols)
            elif code_point >= 32 and code_point < 127:  # Printable ASCII
                # Keep common punctuation, remove special symbols
                if char in '.,;:!?()[]{}':
                    cleaned_chars.append(char)
        
        corrected_text = ''.join(cleaned_chars)
        
        # Normalize Unicode (combine diacritics properly)
        corrected_text = unicodedata.normalize('NFC', corrected_text)
        
        # Clean up extra spaces
        corrected_text = ' '.join(corrected_text.split())
        
        # Re-count Urdu characters after cleaning
        urdu_char_count_after = sum(1 for char in corrected_text if 0x0600 <= ord(char) <= 0x06FF)
        
        # Validation: Check if we have meaningful Urdu content
        # If aggressive cleaning removed too much, use fallback
        if urdu_char_count_after < 2 and urdu_char_count > 0:
            # Use fallback (less cleaned version)
            print(f"Warning: Aggressive cleaning removed too much. Using fallback.")
            corrected_text = fallback_text
            # Light cleaning on fallback
            corrected_text = re.sub(r'[*\\|]', '', corrected_text)
            corrected_text = re.sub(r'\([\d.]+\)', '', corrected_text).strip()
            for corrupted, correct in char_fixes.items():
                corrected_text = corrected_text.replace(corrupted, correct)
            corrected_text = unicodedata.normalize('NFC', corrected_text)
            corrected_text = ' '.join(corrected_text.split())
            urdu_char_count_after = sum(1 for char in corrected_text if 0x0600 <= ord(char) <= 0x06FF)
        
        # Final validation - need at least some Urdu content or reasonable length
        print("=" * 80)
        print("📊 FINAL VALIDATION:")
        print(f"   Cleaned text length: {len(corrected_text)}")
        print(f"   Urdu character count: {urdu_char_count_after}")
        print(f"   Cleaned text: '{corrected_text[:200]}...'")
        print("=" * 80)
        
        # Enhanced validation: Check for meaningful content
        # Allow single punctuation only if it's clearly the model's response (very rare)
        is_just_punctuation = len(corrected_text.strip()) <= 2 and all(c in '۔،؟!.,;:!?' or c.isspace() for c in corrected_text.strip())
        
        if is_just_punctuation:
            print("⚠️ Model returned only punctuation. Checking raw response for actual content...")
            # Check if there's more content in the raw response that we might have missed
            # Look for Urdu text anywhere in the raw response
            urdu_in_raw = sum(1 for c in raw_response if 0x0600 <= ord(c) <= 0x06FF)
            if urdu_in_raw > 5:
                print(f"⚠️ Found {urdu_in_raw} Urdu characters in raw response. Prompt removal may have been too aggressive.")
                # Try a different approach: look for the last occurrence of common prompt endings
                # and take everything after that
                prompt_end_markers = ["لکھیں:", "Output:", "جملہ لکھیں:", "صحیح ترین"]
                for marker in prompt_end_markers:
                    if marker in raw_response:
                        marker_idx = raw_response.rfind(marker)
                        if marker_idx != -1:
                            after_marker = raw_response[marker_idx + len(marker):].strip()
                            urdu_after = sum(1 for c in after_marker if 0x0600 <= ord(c) <= 0x06FF)
                            if urdu_after > 2:
                                corrected_text = after_marker
                                # Re-clean this new text
                                corrected_text = re.sub(r'[*\\|]', '', corrected_text)
                                corrected_text = re.sub(r'\([\d.]+\)', '', corrected_text).strip()
                                corrected_text = ' '.join(corrected_text.split())
                                urdu_char_count_after = sum(1 for char in corrected_text if 0x0600 <= ord(char) <= 0x06FF)
                                print(f"✅ Found content after marker '{marker}': {len(corrected_text)} chars, {urdu_char_count_after} Urdu chars")
                                break
        
        if len(corrected_text) < 2 or (urdu_char_count_after == 0 and len(corrected_text) < 5) or is_just_punctuation:
            print("❌ VALIDATION FAILED: Response too short, no Urdu content, or only punctuation")
            print(f"   Raw response (first 1000 chars): {raw_response[:1000]}")
            print(f"   After prompt removal: {corrected_text[:500]}")
            print(f"   Original response length: {original_length}")
            print(f"   Urdu chars in raw: {sum(1 for c in raw_response if 0x0600 <= ord(c) <= 0x06FF)}")
            
            # Try one more time with a simpler approach: just take the last 200 chars
            # Sometimes the model response is at the very end
            if len(raw_response) > 100:
                last_part = raw_response[-200:].strip()
                urdu_in_last = sum(1 for c in last_part if 0x0600 <= ord(c) <= 0x06FF)
                if urdu_in_last > 2:
                    corrected_text = last_part
                    corrected_text = re.sub(r'[*\\|]', '', corrected_text)
                    corrected_text = re.sub(r'\([\d.]+\)', '', corrected_text).strip()
                    corrected_text = ' '.join(corrected_text.split())
                    urdu_char_count_after = sum(1 for char in corrected_text if 0x0600 <= ord(char) <= 0x06FF)
                    print(f"✅ Found content in last 200 chars: {len(corrected_text)} chars, {urdu_char_count_after} Urdu chars")
                    
                    # Re-validate
                    if len(corrected_text) >= 2 and urdu_char_count_after > 0:
                        print("✅ VALIDATION PASSED after last-part extraction")
                    else:
                        return jsonify({
                            "error": "Model returned corrupted or empty response",
                            "raw_response": raw_response,  # Full response for debugging
                            "cleaned_length": len(corrected_text),
                            "urdu_char_count": urdu_char_count_after,
                            "cleaned_text_preview": corrected_text[:200],
                            "debug_info": f"After cleaning: length={len(corrected_text)}, urdu_chars={urdu_char_count_after}. Raw response length: {len(raw_response)}. Tried multiple extraction methods."
                        }), 500
                else:
                    return jsonify({
                        "error": "Model returned corrupted or empty response",
                        "raw_response": raw_response,  # Full response for debugging
                        "cleaned_length": len(corrected_text),
                        "urdu_char_count": urdu_char_count_after,
                        "cleaned_text_preview": corrected_text[:200],
                        "debug_info": f"After cleaning: length={len(corrected_text)}, urdu_chars={urdu_char_count_after}. Raw response length: {len(raw_response)}. Model may have generated only punctuation."
                    }), 500
            else:
                return jsonify({
                    "error": "Model returned corrupted or empty response",
                    "raw_response": raw_response,  # Full response for debugging
                    "cleaned_length": len(corrected_text),
                    "urdu_char_count": urdu_char_count_after,
                    "cleaned_text_preview": corrected_text[:200],
                    "debug_info": f"After cleaning: length={len(corrected_text)}, urdu_chars={urdu_char_count_after}. Raw response length: {len(raw_response)}"
                }), 500
        
        print("✅ VALIDATION PASSED: Returning cleaned text")
        print("=" * 80)

        return jsonify({
            "status": "success",
            "corrected_text": corrected_text
        })

    except RuntimeError as e:
        # Handle CUDA out of memory errors specifically
        error_str = str(e)
        if "CUDA out of memory" in error_str or "out of memory" in error_str.lower():
            import traceback
            error_details = traceback.format_exc()
            print(f"❌ CUDA Out of Memory Error: {error_details}")
            
            # Try to clear CUDA cache
            try:
                import torch
                torch.cuda.empty_cache()
                print("✅ Cleared CUDA cache")
            except:
                pass
            
            return jsonify({
                "error": "CUDA out of memory",
                "error_type": "RuntimeError",
                "details": error_str,
                "suggestion": "The GPU ran out of memory. Try reducing max_new_tokens or restart the kernel to free GPU memory."
            }), 500
        else:
            # Other RuntimeErrors
            import traceback
            error_details = traceback.format_exc()
            print(f"Error in generate_correction: {error_details}")
            return jsonify({
                "error": str(e),
                "error_type": type(e).__name__,
                "details": error_details
            }), 500
    except Exception as e:
        import traceback
        error_details = traceback.format_exc()
        print(f"Error in generate_correction: {error_details}")
        return jsonify({
            "error": str(e),
            "error_type": type(e).__name__,
            "details": error_details
        }), 500

# Verify route registration after defining the function
print("=" * 80)
print("📋 VERIFYING ROUTE REGISTRATION:")
print("=" * 80)
try:
    if 'generate_correction' in app.view_functions:
        print("✅ Route '/correct' is REGISTERED successfully!")
        print(f"   Function: {app.view_functions['generate_correction']}")
    else:
        print("❌ WARNING: Route '/correct' is NOT registered!")
        print("   This will cause 404 errors. Please re-run the cell.")
    
    # List all routes
    print("\n📋 All registered routes:")
    for rule in app.url_map.iter_rules():
        print(f"   {rule.rule} [{', '.join(rule.methods)}] -> {rule.endpoint}")
except Exception as e:
    print(f"⚠️ Error checking routes: {e}")
print("=" * 80)

In [ ]:
# ==========================================
# DEBUG: Display Last Model Response
# ==========================================
# Run this cell after making a request to see what the model returned
# This helps debug issues with model responses

# Store the last response for debugging
last_model_response = None
last_full_prompt = None

# You can also manually test the model here:
# test_hypotheses = ["test1", "test2", "test3", "test4"]
# test_prompt = "Your prompt here"
# test_response = chatbot(test_prompt, max_new_tokens=200, do_sample=True, temperature=0.05)
# print("Model Response:", test_response)

print("💡 To see the last model response, check the Flask server output above.")
print("💡 Or make a request from the frontend and check the console output.")


In [ ]:
@app.route('/', methods=['GET'])
def health_check():
    """Health check endpoint to verify server is running"""
    # List all registered routes for debugging
    routes = []
    try:
        for rule in app.url_map.iter_rules():
            routes.append({
                "endpoint": rule.endpoint,
                "methods": list(rule.methods),
                "path": rule.rule
            })
    except:
        pass
    
    return jsonify({
        "status": "CORAL Backend is Running",
        "routes": routes,
        "message": "Server is healthy. Use POST /correct for corrections."
    }), 200

# Debug endpoint to list all routes
@app.route('/routes', methods=['GET'])
def list_routes():
    """Debug endpoint to list all registered routes"""
    routes = []
    try:
        for rule in app.url_map.iter_rules():
            routes.append({
                "endpoint": rule.endpoint,
                "methods": list(rule.methods),
                "path": rule.rule
            })
    except Exception as e:
        return jsonify({"error": str(e)}), 500
    
    return jsonify({"routes": routes}), 200


In [ ]:
# ==========================================
# RUN SERVER WITH NGROK
# ==========================================
def run_app():
    app.run(port=5000, debug=False, use_reloader=False)

# CRITICAL: Verify routes are registered before starting server
print("=" * 80)
print("🔍 PRE-FLIGHT CHECK: Verifying routes are registered...")
print("=" * 80)

route_ok = False
try:
    if 'generate_correction' in app.view_functions:
        print("✅ Route '/correct' is registered")
        route_ok = True
    else:
        print("❌ ERROR: Route '/correct' is NOT registered!")
        print("   Please re-run the cell that defines @app.route('/correct')")
        print("   The server will NOT start until the route is registered.")
        route_ok = False
    
    # List all registered routes
    print("\n📋 Currently registered routes:")
    for rule in app.url_map.iter_rules():
        print(f"   {rule.rule} [{', '.join(rule.methods)}] -> {rule.endpoint}")
except Exception as e:
    print(f"⚠️ Error checking routes: {e}")
    route_ok = False

print("=" * 80)

if not route_ok:
    print("\n❌ SERVER WILL NOT START - Route registration failed!")
    print("\n📝 INSTRUCTIONS:")
    print("1. Go back to the cell with '@app.route('/correct')")
    print("2. Re-run that cell to register the route")
    print("3. Then come back and run this cell again")
    raise RuntimeError("Route '/correct' is not registered. Please re-run the route definition cell.")

# Set Auth Token
if NGROK_AUTH_TOKEN == "NGROK_AUTH_TOKEN":
    print("❌ ERROR: You forgot to set your Ngrok Auth Token at the top of the script!")
else:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    public_url = ngrok.connect(5000).public_url
    print(f"\n🚀 BACKEND IS LIVE! Copy this URL for your Frontend:\n")
    print(f"👉 {public_url} 👈\n")
    print(f"\n📋 Available endpoints:")
    print(f"   GET  {public_url}/          - Health check")
    print(f"   GET  {public_url}/routes    - List all routes")
    print(f"   POST {public_url}/correct    - Generate correction")
    print("\n" + "=" * 80)

    # Start Flask in a separate thread
    threading.Thread(target=run_app, daemon=True).start()
    print("✅ Flask server started in background thread")
    print("=" * 80)